<a href="https://colab.research.google.com/github/ollihansen90/Mathe-SH-CT/blob/main/MatheSH_CT_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/ollihansen90/Mathe-SH-CT.git
import os
os.chdir("Mathe-SH-CT")

# 🧠 Computertomographie (CT) – Wie sieht man ins Innere?

Stell dir vor, du möchtest herausfinden, wie etwas von innen aussieht –  
zum Beispiel ein menschlicher Körper oder ein Stück Gepäck am Flughafen.

👉 Aufschneiden wäre keine gute Idee.  
👉 Fotografieren hilft auch nicht, weil man nur die Oberfläche sieht.

Hier kommt die **Computertomographie (CT)** ins Spiel!

---

## 🔍 Was ist Computertomographie?

Die Computertomographie ist ein Verfahren, mit dem man das **Innere eines Objekts sichtbar machen kann**, ohne es zu zerstören.

Dazu werden viele **Röntgenbilder aus verschiedenen Richtungen** aufgenommen.

Diese Bilder nennt man **Projektionen**.

---

## 💡 Die Idee dahinter

Jede Projektion zeigt nicht das ganze Bild, sondern nur eine Art „Schatten“ des Objekts aus einer bestimmten Richtung.

👉 Ein einzelner Schatten reicht nicht aus  
👉 Aber viele Schatten aus verschiedenen Winkeln enthalten zusammen genug Information!

Die große Frage ist:

> **Wie kann man aus vielen Schatten wieder das ursprüngliche Bild zusammensetzen?**

---

## 🏥 Wo wird CT verwendet?

- Medizin (z. B. um Knochen oder Organe zu untersuchen)
- Flughäfen (Gepäckkontrolle)
- Industrie (Materialprüfung)

---

## 🎯 Ziel dieses Notebooks

In diesem Notebook wirst du:

- selbst Projektionen erzeugen
- ein sogenanntes **Sinogramm** berechnen
- versuchen, daraus das ursprüngliche Bild zu rekonstruieren
- und herausfinden, warum das gar nicht so einfach ist 😄

Am Ende baust du Schritt für Schritt eine echte CT-Rekonstruktion nach!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from imageio.v2 import imread
import loesung
from loesung import drehe_bild


## 🖼️ Unser Ausgangsbild

Bevor wir mit der Computertomographie starten, laden wir ein Bild, das wir später „durchleuchten“ wollen.

Dabei passiert Folgendes:

- Das Bild wird von der Datei `bw.png` geladen  
- Die Pixelwerte werden in Zahlen umgewandelt  
- Die Werte werden auf den Bereich **0 bis 1** skaliert  
- Anschließend runden wir die Werte (für ein klares Schwarz-Weiß-Bild)

Zum Schluss zeigen wir das Bild an.

👉 Dieses Bild ist unser "Objekt", das wir später rekonstruieren möchten!

In [ ]:
img = imread("bw.png")[::1,::1]
img = img.astype(float)
img /= np.max(img)
img = np.round(img)

plt.figure()
plt.imshow(img, cmap="gray")
plt.show()

## 🔄 Aufgabe 1: Drehe das Bild!

Jetzt wird es interaktiv 😊

Wir können unser Bild mit der Funktion `drehe_bild` drehen.  
Dazu geben wir einfach das Bild und einen Winkel (in Grad) an.

---

### 🧪 Deine Aufgabe

- Ändere den Winkel (z. B. 0°, 30°, 90°, 180°)
- Beobachte, was mit dem Bild passiert

👉 Fragen zum Nachdenken:

- Was passiert bei 90°?
- Sieht 180° anders aus als das Original?
- Was passiert bei ungewöhnlichen Winkeln wie 37°?

---

💡 Tipp:  
Kleine Änderungen am Winkel können schon einen großen Unterschied machen!

In [ ]:
bild_gedreht = drehe_bild(img, 45)

plt.figure()
plt.imshow(bild_gedreht, cmap="gray")
plt.show()

## 📡 Aufgabe 2: Baue ein Sinogramm

Jetzt machen wir etwas, das eng mit echter Computertomographie zusammenhängt!

---

### 💡 Die Idee

Für einen bestimmten Winkel passiert Folgendes:

1. Das Bild wird gedreht  
2. Dann wird jede Spalte aufsummiert  
   → Wir erhalten eine einzelne Zeile

Diese Zeile ist eine sogenannte **Projektion** (eine Art „Schatten“ des Bildes).

👉 Viele solcher Projektionen aus verschiedenen Winkeln ergeben zusammen ein **Sinogramm**.

---

### ⚠️ Wichtig!

Ein echtes CT-Gerät liefert **nicht direkt das fertige Bild**.

👉 Die **Rohdaten eines CTs sind genau solche Projektionen**  
👉 Also im Prinzip: ein **Sinogramm**

Erst danach wird daraus mit cleveren Algorithmen wieder ein Bild berechnet!

---

### 🧪 Deine Aufgabe

Vervollständige die Funktion `sinogrammzeile`:

- Drehe das Bild um den gegebenen Winkel  
- Speichere das gedrehte Bild in `hilfsbild`  
- Summiere anschließend über die Zeilen (das passiert schon im Code)

---

### 🤔 Fragen zum Nachdenken

- Warum summieren wir über eine Richtung?
- Was könnte eine einzelne Zeile im Sinogramm darstellen?
- Was passiert, wenn du die Anzahl der Winkel (`num_winkel`) erhöhst?

---

💡 Tipp:  
Du hast die Funktion `drehe_bild` schon kennengelernt – die brauchst du hier wieder!

In [ ]:
def sinogrammzeile(bild, winkel):
    # TODO: Zeile des Sinogramms berechnen
    hilfsbild = np.zeros_like(bild)

    return np.sum(hilfsbild, axis=0)

def sinogramm(bild, num_winkel):
    output = []
    for winkel in np.linspace(0,180,num_winkel,endpoint=False):
        output.append(sinogrammzeile(bild, winkel))
    return output

img_sin = sinogramm(img, 18)
plt.figure()
plt.imshow(img_sin, cmap="gray")
plt.show()

## 🔙 Rekonstruktion: Vom Sinogramm zurück zum Bild

Jetzt versuchen wir das Gegenteil von vorher:

👉 Aus vielen Projektionen (dem Sinogramm) wollen wir wieder ein Bild erzeugen!

Dieses Verfahren nennt man **Rückprojektion (Backprojection)**.

---

### 💡 Die Idee der Rückprojektion

Wir haben gesehen:

- Jede Zeile im Sinogramm ist ein "Schatten" des Bildes aus einem bestimmten Winkel

Jetzt machen wir das rückwärts:

1. Wir nehmen eine Zeile aus dem Sinogramm  
2. Wir machen daraus ein sogenanntes **Streifenbild**  
   → Die Werte werden über das ganze Bild "verteilt"  
3. Dieses Streifenbild wird zurückgedreht  
4. Wir addieren alle Streifenbilder auf

👉 Am Ende entsteht wieder ein Bild!

---

### ⚠️ Wichtig zu verstehen

- Eine einzelne Projektion enthält **nicht genug Information**  
- Erst viele Projektionen zusammen ergeben ein brauchbares Bild  
- Die einfache Rückprojektion führt oft zu einem **verschwommenen Ergebnis**

---

### 🧪 Nächster Schritt

Jetzt bauen wir genau so ein **Streifenbild** aus einer einzelnen Sinogramm-Zeile.

👉 Dieses Streifenbild ist ein Baustein für die spätere Rekonstruktion!

In [ ]:
def streifenbild(zeile, winkel):
    # TODO: Streifenbild erstellen
    hilfsbild = np.zeros((len(zeile), len(zeile)))
    return hilfsbild

w = 45
winkel = w//180*len(img_sin)

plt.figure()
plt.imshow(streifenbild(img_sin[winkel], winkel), cmap="gray")
plt.show()

## 🧩 Aufgabe 3: Setze die Streifenbilder wieder zusammen

Jetzt kommt der eigentliche Rekonstruktionsschritt:

Aus dem ganzen Sinogramm soll wieder ein Bild entstehen.

---

### 💡 So funktioniert die Rückprojektion

Für jede Zeile im Sinogramm:

1. Erzeuge ein passendes **Streifenbild**
2. Drehe es in die richtige Richtung
3. Addiere es zum Gesamtbild

Das machen wir für **alle** Projektionen.

Am Ende teilen wir noch durch die Anzahl der Winkel, damit die Werte nicht zu groß werden.

---

### 🧪 Deine Aufgabe

Vervollständige die Schleife in `backprojection`:

- Erzeuge aus `zeile` und dem passenden Winkel ein Streifenbild
- Addiere dieses Streifenbild zu `output`

---

#### 🤔 Beobachte danach genau das Ergebnis

Vergleiche:

- das Originalbild
- das Sinogramm
- die rekonstruierte Version

Überlege dir:

- Was klappt schon gut?
- Was fällt dir an der Rekonstruktion auf?
- Sieht das Ergebnis genauso aus wie das Original?

---

💡 Tipp:  
Du brauchst hier die Funktion `streifenbild`, die du gerade vorbereitet hast.

In [ ]:
def backprojection(sinogramm):
    output = np.zeros((len(sinogramm[0]), len(sinogramm[0])))
    winkel = np.linspace(0,180,len(sinogramm),endpoint=False)

    for i, zeile in enumerate(sinogramm):
        # TODO: Streifenbild erstellen und zum Output addieren (aktuell nur Nullen)
        output += np.zeros((len(zeile), len(zeile)))


    return output/len(sinogramm)

img_reco = backprojection(img_sin)
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.subplot(1,3,2)
plt.imshow(img_sin, cmap="gray")
plt.subplot(1,3,3)
plt.imshow(img_reco, cmap="gray")
plt.show()

## ✨ Verbesserung: Gefilterte Rückprojektion

Im vorherigen Schritt hast du gesehen:

👉 Die Rekonstruktion funktioniert – aber das Bild ist ziemlich verschwommen.

Warum passiert das?

Bei der einfachen Rückprojektion werden bestimmte Bildanteile zu stark "verschmiert".  
Vor allem feine Details gehen dabei verloren.

---

### 💡 Die Idee

Bevor wir die Rückprojektion durchführen, verändern wir das Sinogramm ein wenig:

👉 Wir **filtern jede einzelne Projektion**, bevor wir sie zurückprojizieren.

Das nennt man:

**Filtered Backprojection (gefilterte Rückprojektion)**

---

### 🔍 Was passiert beim Filtern?

Ganz grob gesagt:

- Wichtige Details (feine Strukturen) werden verstärkt  
- "Überflüssige" Anteile werden abgeschwächt  

Dadurch wird das Ergebnis später deutlich schärfer.

👉 Wie genau das mathematisch funktioniert (FFT), ist hier nicht so wichtig. Das lernst du später im Studium.

---

### 🧪 Beobachtung

Vergleiche die beiden Ergebnisse:

- Rekonstruktion **ohne Filter**
- Rekonstruktion **mit Filter**

👉 Achte besonders auf:

- Schärfe der Kanten  
- Details im Bild  
- Allgemeine Bildqualität  

---

### 🤔 Frage

Warum könnte es sinnvoll sein, das Sinogramm **vor** der Rückprojektion zu verändern?

---

💡 Fazit:  
Mit dem richtigen Trick kann man aus denselben Daten ein deutlich besseres Bild rekonstruieren!

In [ ]:
def filter_sinogramm(sinogramm, kernel):
    output = []
    for zeile in sinogramm:
        zeile_fft = np.fft.fft(zeile)
        freqs = np.fft.fftfreq(len(zeile))
        filter = kernel(freqs)
        zeile_fft *= filter
        output.append(np.real(np.fft.ifft(zeile_fft)))
    return output

def ramp_filter(freqs):
    filter = np.abs(freqs)
    return filter

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.subplot(1,3,2)
plt.imshow(img_sin, cmap="gray")
plt.subplot(1,3,3)
plt.imshow(img_reco, cmap="gray")
plt.suptitle("No Filter")
plt.show()

img_sin_filt = filter_sinogramm(img_sin, ramp_filter)
img_reco_filt = backprojection(img_sin_filt)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.subplot(1,3,2)
plt.imshow(img_sin_filt, cmap="gray")
plt.subplot(1,3,3)
plt.imshow(img_reco_filt, cmap="gray")
plt.suptitle(filter.__name__)
plt.show()

## 🧠 Abschluss: Ein echter CT-Scan

Jetzt wenden wir alles, was du gelernt hast, auf ein echtes Beispiel an!

👉 Statt eines einfachen Testbilds verwenden wir nun einen **echten CT-Scan**.

---

### 🔍 Was passiert hier?

- Das CT-Bild wird geladen (leicht verkleinert für schnellere Berechnung)  
- Wir erzeugen daraus ein Sinogramm  
- Wir rekonstruieren das Bild mit:
  - einfacher Rückprojektion  
  - gefilterter Rückprojektion  

---

### 🧪 Experimentiere selbst!

Ein wichtiger Parameter ist:

👉 `num_winkel` = Anzahl der Projektionen

Ändere diesen Wert und beobachte:

- Wie verändert sich das Sinogramm?
- Wie gut wird die Rekonstruktion?

---

#### 🤔 Fragen zum Nachdenken

- Was passiert bei **sehr wenigen Winkeln** (z. B. 10)?
- Was passiert bei **vielen Winkeln** (z. B. 100 oder mehr)?
- Wird das Bild automatisch perfekt, wenn man genug Winkel nimmt?

In [ ]:
img = imread("ct_scan.png")[::5,::5]
img = img.astype(float)
img /= np.max(img)

num_winkel = 10

img_sin = sinogramm(img, num_winkel)
img_reco = backprojection(img_sin)


plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.subplot(1,3,2)
plt.imshow(img_sin, cmap="gray")
plt.subplot(1,3,3)
plt.imshow(img_reco, cmap="gray")
plt.suptitle("Ramp Filter")
plt.show()

img_sin_filt = filter_sinogramm(img_sin, ramp_filter)
img_reco_filt = backprojection(img_sin_filt)
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(img, cmap="gray")
plt.subplot(1,3,2)
plt.imshow(img_sin_filt, cmap="gray")
plt.subplot(1,3,3)
plt.imshow(img_reco_filt, cmap="gray")
plt.suptitle("Ramp Filter")
plt.show()

### 💡 Erkenntnis

- Mehr Projektionen → mehr Information → bessere Rekonstruktion  
- Aber: Mehr Daten bedeutet auch mehr Rechenaufwand  

👉 Genau solche Abwägungen spielen auch in echten CT-Systemen eine große Rolle!

---

🎉 **Geschafft!**

Du hast gerade selbst nachvollzogen, wie Computertomographie funktioniert:

- Projektionen erzeugen  
- Sinogramm berechnen  
- Bild rekonstruieren  
- Qualität durch Filter verbessern  

👉 Das ist im Kern genau das, was auch in echten CT-Geräten passiert!